# MVTec AD — Exploratory Data Analysis

This notebook provides an overview of the MVTec Anomaly Detection dataset
for the three categories used in this project: **bottle**, **hazelnut**, and **carpet**.

We examine:
- Dataset structure and image counts
- Sample images from each category (good + defective)
- Defect type distribution
- Ground truth mask visualization
- Image statistics

In [ ]:
import sys
sys.path.insert(0, '..')

import os
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

DATA_ROOT = Path('../data/mvtec_ad')
CATEGORIES = ['bottle', 'hazelnut', 'carpet']

## 1. Dataset Structure Overview

In [ ]:
for cat in CATEGORIES:
    cat_dir = DATA_ROOT / cat
    print(f'\n=== {cat.upper()} ===')
    
    # Training images
    train_good = cat_dir / 'train' / 'good'
    train_count = len(list(train_good.glob('*.png'))) if train_good.exists() else 0
    print(f'  Train (good only): {train_count} images')
    
    # Test images by defect type
    test_dir = cat_dir / 'test'
    if test_dir.exists():
        print(f'  Test images:')
        total_test = 0
        for subdir in sorted(test_dir.iterdir()):
            if subdir.is_dir():
                count = len(list(subdir.glob('*.png')))
                total_test += count
                marker = '(normal)' if subdir.name == 'good' else '(defect)'
                print(f'    {subdir.name:20s}: {count:3d} images {marker}')
        print(f'    {"TOTAL":20s}: {total_test:3d} images')

## 2. Sample Images: Good vs Defective

In [ ]:
for cat in CATEGORIES:
    cat_dir = DATA_ROOT / cat
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    fig.suptitle(f'{cat.upper()} - Sample Images', fontsize=16, fontweight='bold')
    
    # Row 1: Good (training) images
    good_dir = cat_dir / 'train' / 'good'
    good_imgs = sorted(good_dir.glob('*.png'))[:4]
    for j, img_path in enumerate(good_imgs):
        img = Image.open(img_path)
        axes[0, j].imshow(img)
        axes[0, j].set_title(f'Good ({img_path.stem})')
        axes[0, j].axis('off')
    axes[0, 0].set_ylabel('Normal', fontsize=14)
    
    # Row 2: Defective test images (one per defect type)
    test_dir = cat_dir / 'test'
    defect_dirs = [d for d in sorted(test_dir.iterdir()) if d.is_dir() and d.name != 'good']
    for j, defect_dir in enumerate(defect_dirs[:4]):
        img_path = sorted(defect_dir.glob('*.png'))[0]
        img = Image.open(img_path)
        axes[1, j].imshow(img)
        axes[1, j].set_title(f'{defect_dir.name}')
        axes[1, j].axis('off')
    axes[1, 0].set_ylabel('Defective', fontsize=14)
    
    # Hide unused axes
    for i in range(2):
        for j in range(max(len(defect_dirs), 4), 4):
            axes[i, j].axis('off')
    
    plt.tight_layout()
    plt.show()

## 3. Defect Type Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, cat in enumerate(CATEGORIES):
    test_dir = DATA_ROOT / cat / 'test'
    counts = {}
    for subdir in sorted(test_dir.iterdir()):
        if subdir.is_dir():
            counts[subdir.name] = len(list(subdir.glob('*.png')))
    
    colors = ['#2ecc71' if k == 'good' else '#e74c3c' for k in counts.keys()]
    axes[idx].bar(counts.keys(), counts.values(), color=colors)
    axes[idx].set_title(f'{cat.upper()} - Test Set', fontsize=14)
    axes[idx].set_ylabel('Number of Images')
    axes[idx].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 4. Ground Truth Mask Visualization

In [ ]:
for cat in CATEGORIES:
    cat_dir = DATA_ROOT / cat
    gt_dir = cat_dir / 'ground_truth'
    test_dir = cat_dir / 'test'
    
    defect_dirs = [d for d in sorted(test_dir.iterdir()) if d.is_dir() and d.name != 'good']
    n_defects = min(len(defect_dirs), 4)
    
    fig, axes = plt.subplots(n_defects, 3, figsize=(12, 4 * n_defects))
    if n_defects == 1:
        axes = axes.reshape(1, -1)
    fig.suptitle(f'{cat.upper()} - Ground Truth Masks', fontsize=16, fontweight='bold')
    
    for i, defect_dir in enumerate(defect_dirs[:n_defects]):
        img_path = sorted(defect_dir.glob('*.png'))[0]
        mask_name = img_path.stem + '_mask.png'
        mask_path = gt_dir / defect_dir.name / mask_name
        
        img = Image.open(img_path)
        axes[i, 0].imshow(img)
        axes[i, 0].set_title(f'{defect_dir.name} - Original')
        axes[i, 0].axis('off')
        
        if mask_path.exists():
            mask = Image.open(mask_path)
            axes[i, 1].imshow(mask, cmap='gray')
            axes[i, 1].set_title('Ground Truth Mask')
        axes[i, 1].axis('off')
        
        # Overlay
        img_np = np.array(img.resize((256, 256)))
        if mask_path.exists():
            mask_np = np.array(mask.resize((256, 256), Image.NEAREST))
            overlay = img_np.copy()
            overlay[mask_np > 128] = [255, 0, 0]
            blended = (0.6 * img_np + 0.4 * overlay).astype(np.uint8)
            axes[i, 2].imshow(blended)
            axes[i, 2].set_title('Overlay')
        axes[i, 2].axis('off')
    
    plt.tight_layout()
    plt.show()

## 5. Image Statistics

In [ ]:
for cat in CATEGORIES:
    train_dir = DATA_ROOT / cat / 'train' / 'good'
    imgs = sorted(train_dir.glob('*.png'))[:10]
    
    sizes = []
    means = []
    stds = []
    
    for img_path in imgs:
        img = np.array(Image.open(img_path).convert('RGB'))
        sizes.append(img.shape[:2])
        means.append(img.mean(axis=(0, 1)))
        stds.append(img.std(axis=(0, 1)))
    
    print(f'\n=== {cat.upper()} ===')
    print(f'  Image size: {sizes[0]} (H x W)')
    print(f'  Channels: 3 (RGB)')
    print(f'  Mean pixel (R,G,B): ({np.mean(means, axis=0)[0]:.1f}, {np.mean(means, axis=0)[1]:.1f}, {np.mean(means, axis=0)[2]:.1f})')
    print(f'  Std  pixel (R,G,B): ({np.mean(stds, axis=0)[0]:.1f}, {np.mean(stds, axis=0)[1]:.1f}, {np.mean(stds, axis=0)[2]:.1f})')